In [1]:

import numpy as np
import pandas as pd

train = pd.read_csv("/kaggle/input/binary-buzz-cyber-labs-ml-25/train.csv")
test = pd.read_csv("/kaggle/input/binary-buzz-cyber-labs-ml-25/test.csv")

X = train.drop(columns=["id", "target"]).values
y = train["target"].values
X_test = test.drop(columns=["id"]).values

mean = X.mean(axis=0)
std = X.std(axis=0) + 1e-8

X = (X - mean) / std
X_test = (X_test - mean) / std

np.random.seed(42)
idx = np.random.permutation(len(X))
split = int(0.8 * len(X))

train_idx, val_idx = idx[:split], idx[split:]
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))
    
def balanced_accuracy(y_true, y_pred):
    TP = np.sum((y_true==1)&(y_pred==1))
    TN = np.sum((y_true==0)&(y_pred==0))
    FP = np.sum((y_true==0)&(y_pred==1))
    FN = np.sum((y_true==1)&(y_pred==0))
    return 0.5*((TP/(TP+FN+1e-8)) + (TN/(TN+FP+1e-8)))

def train_logistic(X, y, lr=0.0015, epochs=4000, l2=0.001):
    n, d = X.shape
    w = np.zeros(d)
    b = 0

    pos_weight = np.sum(y==0) / np.sum(y==1)

    for _ in range(epochs):
        z = X @ w + b
        y_pred = sigmoid(z)

        errors = y_pred - y
        errors[y==1] *= pos_weight

        dw = (1/n) * X.T @ errors + l2*w
        db = (1/n) * np.sum(errors)

        w -= lr * dw
        b -= lr * db

    return w, b

w, b = train_logistic(X_train, y_train)

probs = sigmoid(X_val @ w + b)

best_t, best_ba = 0.5, 0
for t in np.linspace(0.2, 0.8, 41):
    preds = (probs >= t).astype(int)
    ba = balanced_accuracy(y_val, preds)
    if ba > best_ba:
        best_ba, best_t = ba, t

print("Best Threshold:", best_t)
print("Validation Balanced Accuracy:", best_ba)


# FINAL TRAIN ON FULL DATA

w, b = train_logistic(X, y)

test_probs = sigmoid(X_test @ w + b)
test_preds = (test_probs >= best_t).astype(int)

submission = pd.DataFrame({
    "id": test["id"],
    "target": test_preds
})

submission.to_csv("submission.csv", index=False)
print("Submission file ready!")


Best Threshold: 0.5
Validation Balanced Accuracy: 0.661111111028858
Submission file ready!
